# Dataset EDA — Penn Action · Gym288 · Gym99
Exploratory analysis across all three datasets used in the ST-GCN pipeline.

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_DIR = Path('/content/Yolo-ST-GCN')
    REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
    BRANCH   = 'refactor-1'
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)
    os.chdir(str(REPO_DIR))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'scipy'], check=True)

ROOT = Path(os.getcwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Working dir:', ROOT)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. Path configuration
Set paths to `None` to skip that dataset.

In [ ]:
if IN_COLAB:
    PENN_LABELS_DIR   = None   # Penn .mat files not on Colab by default
    GYM288_PKL        = '/content/Gym288-skeleton/gym_288_skeleton.pkl'
    GYM99_PKL         = '/content/Gym99-from-Gym288/gym99_from_gym288.pkl'
else:
    PENN_LABELS_DIR   = 'data/PennAction/labels'   # adjust to your local path
    GYM288_PKL        = 'data/Gym288-skeleton/gym_288_skeleton.pkl'
    GYM99_PKL         = 'data/Gym99-from-Gym288/gym99_from_gym288.pkl'

### Download Gym288 on Colab (skip if already present)

In [ ]:
if IN_COLAB and GYM288_PKL and not Path(GYM288_PKL).exists():
    from huggingface_hub import snapshot_download
    dl = Path(GYM288_PKL).parent
    dl.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id='Lozumi/Gym288-skeleton', repo_type='dataset',
                      local_dir=str(dl), local_dir_use_symlinks=False)
    pkl_candidates = sorted(dl.rglob('*.pkl'))
    GYM288_PKL = str(pkl_candidates[0])
    print('Gym288 pickle:', GYM288_PKL)
else:
    print('Gym288 path:', GYM288_PKL)

---
## 2. Penn Action
8-class exercise dataset. 13 Penn joints + 1 virtual center = 14 joints. `.mat` format.

In [ ]:
penn_df = None
penn_data = penn_labels = penn_flags = penn_frame_counts = None

if PENN_LABELS_DIR and Path(PENN_LABELS_DIR).exists():
    from src.penn_dataset import build_penn_data_tensors, load_mat_index
    from src.config import EXERCISE_CLASSES

    penn_df = load_mat_index(PENN_LABELS_DIR)
    penn_data, penn_labels, penn_flags, penn_frame_counts, _ = build_penn_data_tensors(PENN_LABELS_DIR)
    print(f'Penn  total={len(penn_labels)}  train={( penn_flags==1).sum()}  test={(penn_flags==0).sum()}')
    print(f'Tensor shape: {penn_data.shape}  dtype: {penn_data.dtype}')
else:
    print('Penn Action path not found — skipping.')

In [ ]:
if penn_labels is not None:
    from src.config import EXERCISE_CLASSES, ID_TO_CLASS
    counts = Counter(penn_labels.tolist())
    names  = [ID_TO_CLASS[i] for i in range(len(EXERCISE_CLASSES))]
    vals   = [counts.get(i, 0) for i in range(len(EXERCISE_CLASSES))]

    train_counts = Counter(penn_labels[penn_flags == 1].tolist())
    test_counts  = Counter(penn_labels[penn_flags == 0].tolist())

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Total class distribution
    axes[0].barh(names, vals, color='steelblue')
    for i, v in enumerate(vals):
        axes[0].text(v + 1, i, str(v), va='center', fontsize=9)
    axes[0].set_xlabel('Sample count')
    axes[0].set_title('Penn Action — class distribution')

    # Train vs test per class
    tr = [train_counts.get(i, 0) for i in range(len(EXERCISE_CLASSES))]
    te = [test_counts.get(i, 0)  for i in range(len(EXERCISE_CLASSES))]
    x  = np.arange(len(names))
    w  = 0.4
    axes[1].barh(x - w/2, tr, w, label='train', color='steelblue')
    axes[1].barh(x + w/2, te, w, label='test',  color='coral')
    axes[1].set_yticks(x); axes[1].set_yticklabels(names)
    axes[1].set_xlabel('Sample count')
    axes[1].set_title('Penn Action — train vs test per class')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f'Total: {sum(vals)}  |  classes: {len([v for v in vals if v > 0])} / {len(EXERCISE_CLASSES)}')
    print(f'Min class: {min(vals)}  Max: {max(vals)}  Imbalance ratio: {max(vals)/max(min(vals),1):.1f}x')

In [ ]:
if penn_frame_counts:
    fc = np.array(penn_frame_counts)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(fc, bins=40, color='steelblue', edgecolor='white')
    ax.axvline(64, color='red', linestyle='--', label='TARGET_FRAMES=64')
    ax.set_xlabel('Raw frame count'); ax.set_ylabel('# videos')
    ax.set_title('Penn Action — raw frame count distribution')
    ax.legend()
    plt.tight_layout(); plt.show()
    print(f'Frames  min={fc.min()}  median={int(np.median(fc))}  mean={fc.mean():.1f}  max={fc.max()}')
    print(f'Videos shorter than 64 frames: {(fc < 64).sum()} / {len(fc)}')

In [ ]:
if penn_data is not None:
    # Coordinate range and zero-padding check
    x_coords = penn_data[:, 0]  # (N, T, V, 1)
    y_coords = penn_data[:, 1]
    print('X range: [{:.3f}, {:.3f}]  mean={:.3f}  std={:.3f}'.format(
        x_coords.min(), x_coords.max(), x_coords.mean(), x_coords.std()))
    print('Y range: [{:.3f}, {:.3f}]  mean={:.3f}  std={:.3f}'.format(
        y_coords.min(), y_coords.max(), y_coords.mean(), y_coords.std()))
    zero_frac = (penn_data == 0).mean()
    print(f'Fraction of zero values (occlusion/padding): {zero_frac:.3%}')

In [ ]:
if penn_data is not None:
    from src.config import JOINT_NAMES, PENN_BONES_14

    def draw_skeleton(ax, xy, edges, title='', color='steelblue'):
        for (i, j) in edges:
            ax.plot([xy[i, 0], xy[j, 0]], [xy[i, 1], xy[j, 1]], color=color, lw=1.5)
        ax.scatter(xy[:, 0], xy[:, 1], s=20, zorder=5, color=color)
        ax.set_aspect('equal'); ax.invert_yaxis()
        ax.axis('off'); ax.set_title(title, fontsize=8)

    from src.config import ID_TO_CLASS
    sample_idxs = [np.where(penn_labels == c)[0][0] for c in range(len(EXERCISE_CLASSES))
                   if (penn_labels == c).any()]

    n = len(sample_idxs)
    cols = 4; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 3))
    axes = axes.flatten()

    mid_t = penn_data.shape[2] // 2
    for ax, idx in zip(axes, sample_idxs):
        # shape (2, T, V, 1) → (V, 2) at mid frame
        xy = penn_data[idx, :, mid_t, :, 0].T
        draw_skeleton(ax, xy, PENN_BONES_14, title=ID_TO_CLASS[penn_labels[idx]])
    for ax in axes[n:]:
        ax.axis('off')

    fig.suptitle('Penn Action — one skeleton per class (mid-frame)', fontsize=11)
    plt.tight_layout(); plt.show()

---
## 3. Gym288
288-class gymnastics dataset. 17 COCO joints (+ virtual center = 18 with coco18). Pickle format.

In [ ]:
gym288_raw = None

if GYM288_PKL and Path(GYM288_PKL).exists():
    with open(GYM288_PKL, 'rb') as f:
        gym288_raw = pickle.load(f)

    anns   = gym288_raw['annotations']
    split  = gym288_raw['split']
    train_ids = set(split.get('train', []))
    test_ids  = set(split.get('test',  []))

    labels288  = np.array([int(a['label']) for a in anns])
    frame_dirs = [str(a['frame_dir']) for a in anns]
    flags288   = np.array([1 if fd in train_ids else (0 if fd in test_ids else -1)
                           for fd in frame_dirs], dtype=np.int8)

    raw_frames288 = []
    for a in anns:
        kp = np.asarray(a['keypoint'])
        if kp.ndim == 4: kp = kp[0]   # (1,T,J,2) → (T,J,2)
        raw_frames288.append(kp.shape[0])
    raw_frames288 = np.array(raw_frames288)

    print(f'Gym288  total={len(labels288)}  train={(flags288==1).sum()}  test={(flags288==0).sum()}  unknown={(flags288==-1).sum()}')
    print(f'Classes: {labels288.min()} – {labels288.max()}  unique={len(np.unique(labels288))}')
else:
    print('Gym288 pickle not found — skipping.')

In [ ]:
if gym288_raw is not None:
    counts288 = Counter(labels288.tolist())
    vals288   = sorted(counts288.values(), reverse=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Sorted class sizes
    axes[0].bar(range(len(vals288)), vals288, color='steelblue', width=1.0)
    axes[0].set_xlabel('Class rank (sorted by count)')
    axes[0].set_ylabel('Sample count')
    axes[0].set_title('Gym288 — class size distribution (sorted)')

    # Histogram of class sizes
    axes[1].hist(vals288, bins=30, color='steelblue', edgecolor='white')
    axes[1].set_xlabel('Samples per class')
    axes[1].set_ylabel('# classes')
    axes[1].set_title('Gym288 — histogram of class sizes')

    plt.tight_layout(); plt.show()

    print(f'Samples/class  min={min(vals288)}  median={int(np.median(vals288))}  mean={np.mean(vals288):.1f}  max={max(vals288)}')
    print(f'Imbalance ratio (max/min): {max(vals288)/max(min(vals288),1):.1f}x')
    print(f'Classes with < 10 samples: {sum(1 for v in vals288 if v < 10)}')
    print(f'Classes with < 5 samples:  {sum(1 for v in vals288 if v < 5)}')

In [ ]:
if gym288_raw is not None:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(raw_frames288, bins=50, color='steelblue', edgecolor='white')
    ax.axvline(64, color='red', linestyle='--', label='TARGET_FRAMES=64')
    ax.set_xlabel('Raw frame count'); ax.set_ylabel('# videos')
    ax.set_title('Gym288 — raw frame count distribution')
    ax.legend()
    plt.tight_layout(); plt.show()
    print(f'Frames  min={raw_frames288.min()}  median={int(np.median(raw_frames288))}  mean={raw_frames288.mean():.1f}  max={raw_frames288.max()}')
    print(f'Videos shorter than 64 frames: {(raw_frames288 < 64).sum()} ({(raw_frames288 < 64).mean():.1%})')

In [ ]:
if gym288_raw is not None:
    # Train/test class coverage
    train_classes = set(labels288[flags288 == 1].tolist())
    test_classes  = set(labels288[flags288 == 0].tolist())
    only_train    = train_classes - test_classes
    only_test     = test_classes  - train_classes
    both          = train_classes & test_classes

    print('Train/test class coverage')
    print(f'  Classes in both train & test : {len(both)}')
    print(f'  Classes only in train        : {len(only_train)}')
    print(f'  Classes only in test         : {len(only_test)}')

    # Per-class train vs test ratio
    ratios = []
    for c in sorted(both):
        tr = (labels288[flags288 == 1] == c).sum()
        te = (labels288[flags288 == 0] == c).sum()
        ratios.append(tr / (tr + te))

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(ratios, bins=30, color='steelblue', edgecolor='white')
    ax.axvline(0.8, color='red', linestyle='--', label='80% train')
    ax.set_xlabel('Train fraction per class')
    ax.set_ylabel('# classes')
    ax.set_title('Gym288 — train fraction distribution across classes')
    ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
if gym288_raw is not None:
    # Keypoint coordinate stats on a sample
    sample_ann = gym288_raw['annotations'][0]
    kp = np.asarray(sample_ann['keypoint'], dtype=np.float32)
    if kp.ndim == 4: kp = kp[0]
    print(f'Keypoint array shape: {kp.shape}  (T, 17, 2)')
    print(f'X range: [{kp[:,:,0].min():.1f}, {kp[:,:,0].max():.1f}]')
    print(f'Y range: [{kp[:,:,1].min():.1f}, {kp[:,:,1].max():.1f}]')
    print('Coordinates appear to be in pixel space (not normalized)' if kp.max() > 2 else 'Coordinates appear normalized [0,1]')

---
## 4. Gym99
99-class subset mapped from Gym288. Focuses on class balance and label remapping quality.

In [ ]:
gym99_raw = None

# Try to build from Gym288 if Gym99 pickle doesn't exist yet
if GYM99_PKL and not Path(GYM99_PKL).exists() and GYM288_PKL and Path(GYM288_PKL).exists():
    print('Gym99 pickle not found — building from Gym288...')
    from src.gym99_builder import build_gym99_from_gym288_pickle
    Path(GYM99_PKL).parent.mkdir(parents=True, exist_ok=True)
    stats = build_gym99_from_gym288_pickle(GYM288_PKL, GYM99_PKL)
    print('Build stats:', stats)

if GYM99_PKL and Path(GYM99_PKL).exists():
    with open(GYM99_PKL, 'rb') as f:
        gym99_raw = pickle.load(f)

    anns99   = gym99_raw['annotations']
    split99  = gym99_raw['split']
    train99  = set(split99.get('train', []))
    test99   = set(split99.get('test',  []))

    labels99  = np.array([int(a['label']) for a in anns99])
    dirs99    = [str(a['frame_dir']) for a in anns99]
    flags99   = np.array([1 if fd in train99 else (0 if fd in test99 else -1)
                          for fd in dirs99], dtype=np.int8)

    raw_frames99 = []
    for a in anns99:
        kp = np.asarray(a['keypoint'])
        if kp.ndim == 4: kp = kp[0]
        raw_frames99.append(kp.shape[0])
    raw_frames99 = np.array(raw_frames99)

    print(f'Gym99  total={len(labels99)}  train={(flags99==1).sum()}  test={(flags99==0).sum()}  unknown={(flags99==-1).sum()}')
    print(f'Classes: {labels99.min()} – {labels99.max()}  unique={len(np.unique(labels99))}')
else:
    print('Gym99 pickle not found — skipping.')

In [ ]:
if gym99_raw is not None:
    counts99 = Counter(labels99.tolist())
    vals99   = sorted(counts99.values(), reverse=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].bar(range(len(vals99)), vals99, color='coral', width=1.0)
    axes[0].set_xlabel('Class rank (sorted by count)')
    axes[0].set_ylabel('Sample count')
    axes[0].set_title('Gym99 — class size distribution (sorted)')

    axes[1].hist(vals99, bins=25, color='coral', edgecolor='white')
    axes[1].set_xlabel('Samples per class')
    axes[1].set_ylabel('# classes')
    axes[1].set_title('Gym99 — histogram of class sizes')

    plt.tight_layout(); plt.show()

    print(f'Samples/class  min={min(vals99)}  median={int(np.median(vals99))}  mean={np.mean(vals99):.1f}  max={max(vals99)}')
    print(f'Imbalance ratio (max/min): {max(vals99)/max(min(vals99),1):.1f}x')
    print(f'Classes with < 10 samples: {sum(1 for v in vals99 if v < 10)}')
    print(f'Classes with < 5 samples:  {sum(1 for v in vals99 if v < 5)}')

In [ ]:
if gym99_raw is not None:
    train_classes99 = set(labels99[flags99 == 1].tolist())
    test_classes99  = set(labels99[flags99 == 0].tolist())
    only_train99    = train_classes99 - test_classes99
    only_test99     = test_classes99  - train_classes99
    both99          = train_classes99 & test_classes99

    print('Train/test class coverage')
    print(f'  Classes in both train & test : {len(both99)}')
    print(f'  Classes only in train        : {len(only_train99)}')
    print(f'  Classes only in test         : {len(only_test99)}')

    if only_test99:
        print(f'  WARNING: {len(only_test99)} classes appear in test but never in train!')

### 4.1 Neighbor fallback analysis
Rebuild the Gym288→Gym99 mapping and check how many samples required ±1 label fallback.

In [ ]:
if gym288_raw is not None:
    import urllib.request
    from src.config import GYM288_CATEGORIES_URL, GYM99_CATEGORIES_URL
    from src.gym99_builder import parse_finegym_categories

    text288 = urllib.request.urlopen(GYM288_CATEGORIES_URL).read().decode()
    text99  = urllib.request.urlopen(GYM99_CATEGORIES_URL).read().decode()

    pairs288 = parse_finegym_categories(text288)
    pairs99  = parse_finegym_categories(text99)

    clabel288_to_glabel = dict(pairs288)
    glabel_to_clabel99  = {g: c for c, g in pairs99}

    map_288_to_99 = {}
    for c288, g in clabel288_to_glabel.items():
        if g in glabel_to_clabel99:
            map_288_to_99[c288] = glabel_to_clabel99[g]

    direct = fallback_m1 = fallback_p1 = unmatched = 0
    for ann in gym288_raw['annotations']:
        lbl = int(ann['label'])
        if lbl in map_288_to_99:
            direct += 1
        elif (lbl - 1) in map_288_to_99:
            fallback_m1 += 1
        elif (lbl + 1) in map_288_to_99:
            fallback_p1 += 1
        else:
            unmatched += 1

    total = direct + fallback_m1 + fallback_p1 + unmatched

    print(f'Gym288→Gym99 label mapping breakdown  (total={total})')
    print(f'  Direct match  : {direct:6d}  ({direct/total:.1%})')
    print(f'  Fallback -1   : {fallback_m1:6d}  ({fallback_m1/total:.1%})')
    print(f'  Fallback +1   : {fallback_p1:6d}  ({fallback_p1/total:.1%})')
    print(f'  Unmatched     : {unmatched:6d}  ({unmatched/total:.1%})  ← dropped')

    labels = ['Direct', 'Fallback -1', 'Fallback +1', 'Dropped']
    sizes  = [direct, fallback_m1, fallback_p1, unmatched]
    colors = ['steelblue', 'orange', 'coral', 'lightgray']

    fig, ax = plt.subplots(figsize=(6, 4))
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colors,
        autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
        startangle=140
    )
    ax.set_title('Gym288→Gym99 label mapping breakdown')
    plt.tight_layout(); plt.show()

In [ ]:
if gym288_raw is not None:
    # Which Gym288 classes have no direct Gym99 match at all?
    all288_classes = set(clabel288_to_glabel.keys())
    direct_match_classes = set(map_288_to_99.keys())
    no_match = all288_classes - direct_match_classes
    print(f'Gym288 classes with no direct Gym99 mapping: {len(no_match)} / {len(all288_classes)}')
    print(f'These are dropped entirely unless neighbor fallback finds a hit.')

---
## 5. Cross-dataset comparison

In [ ]:
summary = []

if penn_labels is not None:
    from src.config import EXERCISE_CLASSES
    summary.append({
        'Dataset': 'Penn Action',
        'Total': len(penn_labels),
        'Train': (penn_flags == 1).sum(),
        'Test': (penn_flags == 0).sum(),
        'Classes': len(EXERCISE_CLASSES),
        'Joint layout': 'penn14 (13+1 virtual)',
        'Format': '.mat',
    })

if gym288_raw is not None:
    summary.append({
        'Dataset': 'Gym288',
        'Total': len(labels288),
        'Train': (flags288 == 1).sum(),
        'Test': (flags288 == 0).sum(),
        'Classes': len(np.unique(labels288)),
        'Joint layout': 'coco18 (17+1 virtual)',
        'Format': '.pkl',
    })

if gym99_raw is not None:
    summary.append({
        'Dataset': 'Gym99',
        'Total': len(labels99),
        'Train': (flags99 == 1).sum(),
        'Test': (flags99 == 0).sum(),
        'Classes': len(np.unique(labels99)),
        'Joint layout': 'coco18 (17+1 virtual)',
        'Format': '.pkl (built from Gym288)',
    })

if summary:
    df = pd.DataFrame(summary).set_index('Dataset')
    display(df)

In [ ]:
available = []
if gym288_raw is not None: available.append(('Gym288', vals288, 'steelblue'))
if gym99_raw  is not None: available.append(('Gym99',  vals99,  'coral'))

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 4), sharey=False)
    if len(available) == 1: axes = [axes]

    for ax, (name, vals, color) in zip(axes, available):
        ax.hist(vals, bins=30, color=color, edgecolor='white')
        ax.axvline(np.mean(vals), color='black', linestyle='--', label=f'mean={np.mean(vals):.0f}')
        ax.set_xlabel('Samples per class')
        ax.set_ylabel('# classes')
        ax.set_title(f'{name} — class size histogram')
        ax.legend()

    plt.tight_layout(); plt.show()

In [ ]:
available_frames = []
if penn_frame_counts: available_frames.append(('Penn Action', np.array(penn_frame_counts), 'steelblue'))
if gym288_raw is not None: available_frames.append(('Gym288', raw_frames288, 'darkorange'))
if gym99_raw  is not None: available_frames.append(('Gym99',  raw_frames99,  'coral'))

if available_frames:
    fig, ax = plt.subplots(figsize=(9, 4))
    for name, fc, color in available_frames:
        ax.hist(fc, bins=50, alpha=0.6, label=name, color=color, edgecolor='none')
    ax.axvline(64, color='red', linestyle='--', linewidth=1.5, label='TARGET_FRAMES=64')
    ax.set_xlabel('Raw frame count')
    ax.set_ylabel('# videos')
    ax.set_title('Frame count distribution across datasets')
    ax.legend()
    plt.tight_layout(); plt.show()

    for name, fc, _ in available_frames:
        print(f'{name:12s}  min={fc.min():4d}  median={int(np.median(fc)):4d}  mean={fc.mean():.1f}  max={fc.max()}  short(<64): {(fc<64).mean():.1%}')